# DNABERT (Original) -- Geometric Stability Experiment

Tests geometric stability across the **4 k-mer variants** of the original DNABERT.

## Why This Matters

- **DNABERT-2** uses BPE tokenization; **DNABERT (original)** uses k-mer tokenization
- 4 variants (3-mer, 4-mer, 5-mer, 6-mer) all ~86-90M params but different tokenizations
- This controls for **tokenization strategy** while keeping architecture constant
- If stability varies across k-mer sizes, tokenization granularity matters for geometry

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU
3. Run cells in order

---

In [ ]:
# Install Dependencies

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'

OUTPUT_BASE = './results/dnabert_scaling_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 510,  # DNABERT max_position_embeddings=512
        'models': [
            'zhihan1996/DNA_bert_3',   # 3-mer, ~86M
            'zhihan1996/DNA_bert_6',   # 6-mer, ~90M
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 510,
        'models': [
            'zhihan1996/DNA_bert_3',   # 3-mer, ~86M
            'zhihan1996/DNA_bert_4',   # 4-mer, ~87M
            'zhihan1996/DNA_bert_5',   # 5-mer, ~88M
            'zhihan1996/DNA_bert_6',   # 6-mer, ~90M
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: DNABERT (original, k-mer tokenization)")
print(f"Sequences: {config['n_sequences']}")
print(f"Sequence length: {config['seq_length']} nucleotides")
print(f"Models: {len(config['models'])} (k-mer variants)")
print(f"Note: All ~86-90M params; varying tokenization, not size")

In [ ]:
# Generate Synthetic DNA Sequences

import numpy as np

DNA_BASES = ['A', 'C', 'G', 'T']

def generate_dna_sequences(n_sequences, seq_length, seed=320):
    rng = np.random.default_rng(seed)
    sequences = [
        ''.join(rng.choice(DNA_BASES, size=seq_length))
        for _ in range(n_sequences)
    ]
    print(f"Generated {len(sequences)} DNA sequences (length={seq_length})")
    print(f"Sample: {sequences[0][:50]}...")
    return sequences

sequences = generate_dna_sequences(config['n_sequences'], config['seq_length'])

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''

def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    n_mutations = max(1, int(len(seq) * mutation_rate))
    positions = rng.choice(len(seq), size=n_mutations, replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)

def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))

class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results

suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
test_perturbed = suite.run_all(sequences[:5])
for name, pset in test_perturbed.items():
    dists = [sum(a != b for a, b in zip(o, p)) / len(o)
             for o, p in zip(sequences[:5], pset.sequences)]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# DNABERT Model Wrapper
#
# Original DNABERT uses k-mer tokenization: the sequence is split into
# overlapping k-mers before being fed to the tokenizer.
# e.g., for 6-mer: ATCGATCG -> 'ATCGAT TCGATC CGATCG'

import torch
import gc
import re
from transformers import AutoTokenizer, AutoModel


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def seq_to_kmer(sequence, k):
    """Convert a DNA sequence to space-separated k-mers."""
    return ' '.join(sequence[i:i+k] for i in range(len(sequence) - k + 1))


def get_kmer_size(model_name):
    """Extract k-mer size from model name (DNA_bert_3 -> 3)."""
    match = re.search(r'bert_(\d+)', model_name)
    return int(match.group(1)) if match else 6


def make_dnabert_embedding_fn(model_name, batch_size=8):
    """Load original DNABERT and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    k = get_kmer_size(model_name)
    print(f"  K-mer size: {k}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_name, trust_remote_code=True,
    ).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            # Convert to k-mer representation
            kmer_seqs = [seq_to_kmer(s, k) for s in batch_seqs]

            tokens = tokenizer(
                kmer_seqs, return_tensors='pt', padding=True,
                truncation=True, max_length=512,
            )
            tokens = {key: v.to(device) for key, v in tokens.items()}

            outputs = model(**tokens)
            hidden = outputs.last_hidden_state

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("DNABERT wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0, metric='cosine', n_splits=30,
    seed=320, max_samples=2500, n_bootstrap=config['n_bootstrap'],
)
print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os, time, pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'DNABERT (ORIGINAL) K-MER EXPERIMENT -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_dnabert_embedding_fn(
        model_name, config['batch_size'])
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_')
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name, embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings)
    reports.append(report)

    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name, 'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite: {summary["mean_composite_stability"]:.4f}')

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/dnabert_scaling_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved to {detailed_path}')
print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization -- K-mer Comparison

import matplotlib.pyplot as plt

summaries = [r.summary() for r in reports]
kmer_sizes = [get_kmer_size(r.model_name) for r in reports]
composites = [s['mean_composite_stability'] for s in summaries]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(kmer_sizes, composites, color=['tab:blue', 'tab:green', 'tab:orange', 'tab:red'][:len(kmer_sizes)],
       width=0.6)
ax.set_xlabel('K-mer Size', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('DNABERT: Effect of K-mer Tokenization on Geometric Stability', fontweight='bold')
ax.set_xticks(kmer_sizes)
ax.grid(True, alpha=0.3, axis='y')

for i, (k, v) in enumerate(zip(kmer_sizes, composites)):
    ax.text(k, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dnabert_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-Experiment Comparison

import matplotlib.pyplot as plt, pandas as pd, glob

fig, ax = plt.subplots(figsize=(10, 6))

# DNABERT k-mers as separate points (all ~86-90M)
for i, (k, comp) in enumerate(zip(kmer_sizes, composites)):
    ax.scatter([model_param_counts[i]], [comp], s=200, zorder=5,
               label=f'DNABERT {k}-mer')

for label, pattern, color, marker in [
    ('DNABERT-2 (BPE)', '**/dnabert2_scaling_*_detailed.csv', 'tab:green', 'P'),
    ('NT (Transformer)', '**/nt_scaling_*_detailed.csv', 'tab:orange', 's-'),
    ('ESM-2 (Transformer)', '**/esm2_scaling_*_detailed.csv', 'tab:red', 'o-'),
    ('Caduceus (SSM)', '**/caduceus_scaling_*_detailed.csv', 'tab:blue', 'D-'),
]:
    files = glob.glob(f'{RESULTS_DIR}/../../../{pattern}', recursive=True)
    if files:
        df = pd.read_csv(files[0])
        agg = df.groupby('size_M')['composite'].mean().reset_index()
        ax.plot(agg['size_M'], agg['composite'], marker,
                color=color, linewidth=2, markersize=10, label=label)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('DNABERT K-mers in Cross-Architecture Context', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dnabert_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()